# Global 21 cm signal with and without X-ray heating

Lumina (500 cMpc), Lumina without X-ray heating (option 3, every gas cell, whole box) and the 50 cMpc pair Lumina-50 / Lumina-50-NoX (Zier et al. 2026, App. B), all at gas-cell level.

Inputs (on ORCD, run from `xray/`): `data/lumina_noX_global.csv` from `python lumina_noX.py summary`, which collects
- Lumina X and no X: the whole-box no-X scans (`lumina_noX.py scan --whole-box`, then `merge`): every cell, with the H I-weighted T and T_S sums (2026-11-24);
- the pair: `pair_T_TS.py` (`cells_TTS_NNN.npz`, T and T_S of every cell) and `cells_Tb_xray.py` (`cells_Tb_NNN.npz`, box ⟨δT_b⟩).

Temperatures are H I-weighted harmonic means over all H I (every cell, weight X_H m (1 − x_HII), star-forming gas neutral; user, 2026-11-24): T_gas = ⟨1/T⟩⁻¹, T_S = ⟨1/T_S⟩⁻¹ = T_γ / (1 − ⟨1 − T_γ/T_S⟩). These are the temperatures that set the global δT_b.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

mpl.rcParams['figure.dpi'] = 200
mpl.rcParams['savefig.dpi'] = 300
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
})

In [ ]:
# ---- data: lumina_noX.py summary (whole box, gas cells) ----
CSV = "data/lumina_noX_global.csv"
ZMAX = 16.0                                    # x-axis cut (user, 2026-11-24): drops the X-only z = 23, 18.3 points
d = pd.read_csv(CSV, comment="#").sort_values("z", ascending=False)
d = d[d["z"] <= ZMAX].reset_index(drop=True)
SEL = "_all"            # all H I (user, 2026-11-24); "" = neutral non-SF gas only (x_HII < 1e-3)
need = [f"{c}{SEL}" for c in ["T_X", "T_noX", "TS_X", "TS_noX", "pair_T_X", "pair_T_noX", "pair_TS_X", "pair_TS_noX"]]
missing = [c for c in need if c not in d or not np.isfinite(d[c]).any()]
if missing:
    print(f"no values yet for {missing}: rerun the whole-box scans / pair_T_TS.py, then lumina_noX.py summary")
old = np.isfinite(d["noX_box_est"]) & (d["X_box_src"] != "cells_box")      # X-only rows (z = 23, 18.3) are gas cells
if old.any():
    print("no-X rows not from the whole-box scan:", dict(zip(d["snap"][old], d["X_box_src"][old])))

z = d["z"].to_numpy()
zz = np.linspace(ZMAX, z.min() - 0.3, 300)
T_cmb = 2.7255 * (1.0 + zz)

# colour = run, solid = gas temperature, dashed = spin temperature
RUNS = [  # label, column suffix, colour
    ("Lumina",                 "X",   "#1f4e9c"),
    ("Lumina, no X-rays",      "noX", "#c0471d"),
    ("Lumina-50",              "X",   "#7fa8e0"),
    ("Lumina-50-NoX",          "noX", "#f0a274"),
]
pref = ["", "", "pair_", "pair_"]
tb_col = {("", "X"): "X_box", ("", "noX"): "noX_box_est", ("pair_", "X"): "pair_X", ("pair_", "noX"): "pair_noX"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8.5), sharex=True, gridspec_kw=dict(hspace=0.06))

# (a) gas and spin temperatures
ax1.plot(zz, T_cmb, color="k", lw=1.4, label=r"$T_\mathrm{CMB}$")
for (lab, sfx, c), p in zip(RUNS, pref):
    big = p == ""                                   # Lumina: thicker lines
    for col, ls, mfc in ((f"{p}T_{sfx}{SEL}", "-", c), (f"{p}TS_{sfx}{SEL}", "--", "white")):
        ok = np.isfinite(d[col])
        ax1.plot(z[ok], d[col][ok], ls=ls, color=c, lw=2.0 if big else 1.4, marker="o", ms=4.5 if big else 3.5,
                 mfc=mfc, mec=c, mew=1.1)
ax1.set_yscale("log")
ax1.set_ylabel(r"$T\ [\mathrm{K}]$")
ax1.text(0.03, 0.05, "(a)", transform=ax1.transAxes, va="bottom")

# (b) global 21 cm brightness temperature
ax2.axhline(0, color="0.6", lw=0.8)
for (lab, sfx, c), p in zip(RUNS, pref):
    big = p == ""
    col = tb_col[(p, sfx)]
    ok = np.isfinite(d[col])
    ax2.plot(z[ok], d[col][ok], "-", color=c, lw=2.0 if big else 1.4, marker="o", ms=4.5 if big else 3.5, mec=c)
ax2.set_xlabel(r"$z$")
ax2.set_ylabel(r"$\langle \delta T_b \rangle\ [\mathrm{mK}]$")
ax2.text(0.03, 0.93, "(b)", transform=ax2.transAxes, va="top")
ax2.set_xlim(ZMAX, z.min() - 0.3)

# legends: runs (colour) in (b), line style + T_CMB in (a)
h_runs = [Line2D([], [], color=c, lw=2.0 if p == "" else 1.4, marker="o", ms=4, label=lab)
          for (lab, sfx, c), p in zip(RUNS, pref)]
h_kind = [Line2D([], [], color="0.3", ls="-", marker="o", ms=4, label=r"$T_\mathrm{gas} = \langle 1/T \rangle^{-1}_\mathrm{HI}$"),
          Line2D([], [], color="0.3", ls="--", marker="o", ms=4, mfc="white", label=r"$T_S = \langle 1/T_S \rangle^{-1}_\mathrm{HI}$"),
          Line2D([], [], color="k", lw=1.4, label=r"$T_\mathrm{CMB}$")]
ax1.legend(handles=h_kind, loc="upper left", frameon=False, fontsize=11)
ax2.legend(handles=h_runs, loc="lower left", frameon=False, fontsize=11)

fig.savefig("data/global_T_TS_Tb.pdf", bbox_inches="tight")
fig.savefig("data/global_T_TS_Tb.png", bbox_inches="tight")
plt.show()

# the numbers behind the figure
cols = ["snap", "z", "Tgamma"] + need + ["X_box", "noX_box_est", "pair_X", "pair_noX"]
d[[c for c in cols if c in d]].round(3)